# Alpha Zero Divided Modules Test


## Import Game Env

In [45]:
import sys
import os
import argparse

# Instead of __file__, use os.getcwd()
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

from src.envs import N3il

In [46]:
n=3

current_dir = os.getcwd()

args = {
    'environment': 'N3il',  # Specify the environment
    'algorithm': 'MCTS',
    'max_level_to_use_symmetry': -1,  # Use symmetry for first 2 levels (helps find compact solutions)
    'n': n,
    'C': 1.41,  # 1e-7 for n=20
    'num_searches': 100*(n**2),  # Adjusted for larger n
    'num_workers': 1,      # >1 ⇒ parallel
    'virtual_loss': 1.0,     # magnitude to subtract at reservation
    'process_bar': True,
    'display_state': True,
    'logging_mode': True,  # Enable logging mode to get return value
    'TopN': n,  # Without Priority
    "simulate_with_priority": False,
    'table_dir': current_dir,  # Directory to save tables
    'figure_dir': os.path.join(current_dir, 'figure'),  # Directory to save figures
    'random_seed': 1,  # Use the loop index as a seed for reproducibility
    'tree_visualization': False,  # Set to True to enable tree visualization
}

n3il_test = N3il((n, n), args)

## ResNet

In [47]:
# --- Torch / Device checks for macOS (Apple Silicon M4) ---
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())  # Usually False on Apple Silicon

# Metal (MPS) backend (Apple GPU)
mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
mps_built = hasattr(torch.backends, "mps") and torch.backends.mps.is_built()

print("MPS built:", mps_built)
print("MPS available:", mps_available)

device = torch.device(
    "mps" if mps_available else ("cuda" if torch.cuda.is_available() else "cpu")
)
print("Using device:", device)

# Optional quick sanity test on selected device
try:
    x = torch.randn(2, 2, device=device)
    print("Test tensor sum:", x.sum().item())
except Exception as e:
    print("Device test failed:", e)

torch.manual_seed(0)

Torch version: 2.7.1
CUDA available: False
MPS built: True
MPS available: True
Using device: mps
Test tensor sum: 0.3305668830871582


In [48]:
class ResNet(nn.Module):
    def __init__(self, game, num_resBlocks, num_hidden, device):
        super().__init__()
        self.device = device
        self.startBlock = nn.Sequential(
            nn.Conv2d(2, num_hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_hidden),
            nn.ReLU()
        )

        self.backBone = nn.ModuleList(
            [ResBlock(num_hidden) for i in range(num_resBlocks)]
        )

        self.policyHead = nn.Sequential(
            nn.Conv2d(num_hidden, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * game.row_count * game.column_count, game.action_size)
        )

        self.valueHead = nn.Sequential(
            nn.Conv2d(num_hidden, 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(2),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(2 * game.row_count * game.column_count, 1),
            nn.Tanh()
        )

        self.to(device)

    def forward(self, x):
        x = self.startBlock(x)
        for resBlock in self.backBone:
            x = resBlock(x)
        policy = self.policyHead(x)
        value = self.valueHead(x)
        return policy, value

class ResBlock(nn.Module):
    def __init__(self, num_hidden):
        super().__init__()
        self.conv1 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(num_hidden)
        self.conv2 = nn.Conv2d(num_hidden, num_hidden, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(num_hidden)

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x += residual
        x = F.relu(x)
        return x

## Helper Functions

### Exploration Decay

In [49]:
from numba import njit
import numpy as np

@njit(cache=True, nogil=True)
def exploration_decay_nb(x):  # Monotone-down from (0,1) to (1,0)
    # Cosine decay
    # return (np.cos(np.pi * x)+1)/2  # 100% exploration at start, 0% at end
    
    # Linear
    # return 1 - 0.7 * x   # Found optimal 4-point solution: 86/100 times (86.0%)
    # return 1 - x # 85/100 times (85.0%)

    # Square root (gentle early decay)
    # return 1 - 0.9 * np.sqrt(x) # 91/100 times (91.0%)
    # return 1 - 1 * np.sqrt(x) # 83/100 times (83.0%)
    # return 1 - 0.5 * np.sqrt(x) # 88/100 times (88.0%)
    # return 1 - 0.7 * np.sqrt(x) # 86%
    # return 1 - 0.8 * np.sqrt(x) # 92/100 times (92.0%)
    return 1 - 0.85 * np.sqrt(x)

    # Quadratic (faster decay)
    # return 1 - (x ** 2)

    # Exponential (custom normalization)
    # return ((np.exp(1)/(np.exp(1)-1))**2) * ((np.exp(-x)-np.exp(-1)) ** 2) # 85/100 times (85.0%)

    # Exponential fast (k=3)
    #k = 3.0
    # return (np.exp(-k * x) - np.exp(-k)) / (1 - np.exp(-k)) # 90/100 times (90.0%)

    # Exponential slow (k=1)
    # k = 1.0
    # return (np.exp(-k * x) - np.exp(-k)) / (1 - np.exp(-k)) # 86/100 times (86.0%)

    # Cosine decay
    # return 0.5 * (1 + np.cos(np.pi * x)) # 85/100 times (85.0%)

    # Rational decay
    # a = 1.0
    # return (1 - x) / (1 + a * x) # solution: 90/100 times (90.0%)

    # Logistic decay
    # k = 10.0
    # g0 = 1 / (1 + np.exp(k * (0 - 0.5)))
    # g1 = 1 / (1 + np.exp(k * (1 - 0.5)))
    # gx = 1 / (1 + np.exp(k * (x - 0.5)))
    # return (gx - g1) / (g0 - g1) # 86/100 times (86.0%)

    # Cubic decay
    # return 1 - x ** 3 # 91/100 times (91.0%)
    # return 1 - (0.9 * (x ** 3)) # 91/100 times (91.0%)

### Rewarding Function

In [50]:
from numba import njit
import numpy as np

In [51]:
## Remember Also Adjust get_value_nb in collinear_for_mcts.py !!!!!!!!!!!!!!!!!!!!!!!!!!!!
@njit(cache=True, nogil=True)
def get_value_nb(state, pts_upper_bound):
    total = np.sum(state)
    n = pts_upper_bound/2
    
    # === REVERSE REWARDING FUNCTIONS (prefer smaller point counts) ===
    
    # 1. Simple Linear Inverse: 1.0 for empty board, 0.0 for full board
    # return (1.2*n - total) / n  # Range: [0, 1]
    
    # 2. Exponential Decay (Strong preference for fewer points)
    return np.exp(2.0 * ((total-n) / n))  # Range: [e^-2, 1] ≈ [0.135, 1]
    # return np.exp(-1.0 * (total / n))  # Range: [e^-1, 1] ≈ [0.368, 1]
    # return np.exp(-0.5 * (total / n))  # Range: [e^-0.5, 1] ≈ [0.607, 1]
    
    # 3. Power Functions (Adjustable curvature)
    # return ((n - total) / n) ** 2  # Quadratic preference: [0, 1]
    # return ((n - total) / n) ** 0.5  # Square root preference: [0, 1]
    # return ((n - total) / n) ** 3  # Cubic preference (very aggressive): [0, 1]
    
    # 4. Sigmoid-based (Smooth transition around target)
    # target = n * 0.3  # Target 30% of grid filled
    # return 1.0 / (1.0 + np.exp(0.5 * (total - target)))  # Range: ≈[0, 1]
    # return 1.0 / (1.0 + np.exp(1.0 * (total - target)))  # Steeper transition
    
    # 5. Logarithmic Penalty
    # return max(0, 1.0 - np.log(1.0 + total) / np.log(1.0 + n))  # Range: [0, 1]
    
    # 6. ReLU-based with different thresholds
    # return max(0, (1.2 * n - total) / n)  # Reward up to 120% of n: [0, 1.2]
    # return max(0, (1.5 * n - total) / n)  # Current: reward up to 150% of n
    
    # === OPTIMAL FOR 3x3 MINIMAL COMPLETE SET (4 points) ===
    # Simple linear inverse works best for finding exact minimal sets
    # return (1.6*n - total) * n  / (1.6 - 1.3) # Range: [0, 1], 1.0 for empty, 0.0 for full !!!CURRENT OPTIMAL!!!

    # Baseline rewarding function
    '''
    baseline = 1.6 * n
    theoretical_min = 1.3 * n
    num = baseline - total
    if num > 0:
        return num / (baseline - theoretical_min)  # Range: [0, 1], 1.0 for empty, 0.0 for full
    if num <= 0:
        return num / (baseline - theoretical_min)  # Range: [-1, 0], 0.0 for empty, -1.0 for full
    '''
    # Numba-safe scalar casts
    total = np.float64(np.sum(state))
    n = np.float64(pts_upper_bound) / 2.0

    # Target and normalization
    target = 0.9 * n
    max_possible = 2.0 * n
    eps = np.float64(1e-12)
    span = np.maximum(max_possible - target, eps)  # avoid division by zero
    # Normalized distance: 0 at target, 1 at 2n (can be < 0 if total < target)
    tnorm = (total - target) / span

    # ---- Choose ONE of the following returns (uncomment exactly one) ----

    # 2) Quadratic (penalizes farther from target more strongly)
    # return np.clip(1.0 - tnorm * tnorm, 0.0, 1.0)

    # 3) Gaussian peak at target (default active; sharp pull to 0.9n)
    # sigma = np.maximum(0.05 * n, eps)  # controls sharpness
    # return np.exp(-0.5 * ((total - target) / sigma) ** 2)

    # 4) Logistic decay from target upward
    # k = 6.0 / np.maximum(n, 1.0)
    # return 1.0 / (1.0 + np.exp(k * (total - target)))

    # 5) Rational distance penalty (gentler tail)
    # alpha = 2.0 / np.maximum(n, 1.0)
    # return 1.0 / (1.0 + alpha * np.abs(total - target))

    # 6) Piecewise: full at/below target, then linear drop to 0 at 2n
    # if total <= target:
    #     return 1.0
    # else:
    #     return np.maximum(0.0, 1.0 - (total - target) / span)

    # 7) Cosine half-wave on [target, 2n] (smooth with zero slope at target)
    # x = np.clip(tnorm, 0.0, 1.0)               # map [target,2n] -> [0,1]
    # return 0.5 * (1.0 + np.cos(np.pi * x))     # 1 at target, 0 at 2n

    # ------------ Positive-direction variants (optimum at 2n) ------------
    # Use these if you want to test the opposite objective (larger total better).
    # 1+) Linear increasing from target to 2n
    # return np.clip(tnorm, 0.0, 1.0)

    # 2+) Quadratic increasing (slow start, faster near 2n)
    # x = np.clip(tnorm, 0.0, 1.0)
    # return x * x

    # 3+) Exponential rise (very low until near 2n)
    # x = np.clip(tnorm, 0.0, 1.0)
    # k = 4.0
    # return (np.exp(k * x) - 1.0) / (np.exp(k) - 1.0)

### Simulate

In [52]:
from numba import njit
import numpy as np

@njit(cache=True, nogil=True)
def simulate_nb(state, row_count, column_count, pts_upper_bound):
    """
    Perform random rollout until no valid moves remain.
    Return normalized value using a custom value function.
    Uses get_valid_moves_subset_nb for incremental validity updates.
    Note: This function uses numba's random number generator which is seeded globally.
    """
    max_size = row_count * column_count
    # Initial valid moves mask
    valid_moves = get_valid_moves_nb(state, row_count, column_count)
    total_valid = np.sum(valid_moves)

    while total_valid > 0:
        # Build list of valid actions
        acts = np.empty(total_valid, np.int64)
        k = 0
        for idx in range(max_size):
            if valid_moves[idx]:
                acts[k] = idx
                k += 1
        # Randomly select one valid action and place the point
        pick = acts[np.random.randint(0, total_valid)]

        # Incrementally update valid_moves using subset-based filtering
        valid_moves = get_valid_moves_subset_nb(
            state,
            valid_moves,
            pick,
            row_count,
            column_count
        )

        r = pick // column_count
        c = pick % column_count
        state[r, c] = 1  # mark the new point

        total_valid = np.sum(valid_moves)

    # Compute and return the final value
    return get_value_nb(state, pts_upper_bound)

@njit(cache=True, nogil=True)
def filter_top_priority_moves(valid_moves, priority_grid, row_count, column_count, top_N=1):
    """
    Numba-accelerated: Filter valid moves to only those with the top_N highest priorities.

    Args:
        valid_moves (np.ndarray): 1D array (flattened) of valid moves (1=valid, 0=invalid).
        priority_grid (np.ndarray): 2D array of priority values for each grid cell.
        row_count (int): Number of rows in the grid.
        column_count (int): Number of columns in the grid.
        top_N (int): Number of top priority levels to select.

    Returns:
        np.ndarray: 1D mask array with only the top_N-priority valid moves set to 1.
    """
    indices = []
    priorities = []
    for idx in range(valid_moves.shape[0]):
        if valid_moves[idx] == 1:
            indices.append(idx)
            i = idx // column_count
            j = idx % column_count
            priorities.append(priority_grid[i, j])
    if len(indices) == 0:
        return valid_moves

    # Find the unique priorities and sort descending
    # Numba doesn't support np.unique or sort for lists, so do it manually
    # 1. Copy priorities to a new array
    n = len(priorities)
    unique_priorities = []
    for k in range(n):
        p = priorities[k]
        found = False
        for l in range(len(unique_priorities)):
            if unique_priorities[l] == p:
                found = True
                break
        if not found:
            unique_priorities.append(p)
    # 2. Sort unique_priorities descending (simple selection sort)
    for i in range(len(unique_priorities)):
        max_idx = i
        for j in range(i+1, len(unique_priorities)):
            if unique_priorities[j] > unique_priorities[max_idx]:
                max_idx = j
        # Swap
        tmp = unique_priorities[i]
        unique_priorities[i] = unique_priorities[max_idx]
        unique_priorities[max_idx] = tmp

    # 3. Select top_N priorities
    N = min(top_N, len(unique_priorities))
    threshold = unique_priorities[:N]

    # 4. Build mask
    mask = np.zeros_like(valid_moves)
    for k in range(n):
        idx = indices[k]
        p = priorities[k]
        for t in range(N):
            if p == threshold[t]:
                mask[idx] = 1
                break
    return mask

@njit(cache=True, nogil=True)
def simulate_with_priority_nb(state, row_count, column_count, pts_upper_bound, priority_grid, top_N):
    """
    Perform a random rollout that first filters valid moves by priority
    and then proceeds like simulate_nb, but initial valid moves are pre-filtered.
    Args:
        state (np.ndarray): 2D board state.
        row_count (int): Number of rows.
        column_count (int): Number of columns.
        pts_upper_bound (int): Scoring upper bound.
        priority_grid (np.ndarray): 2D array of priorities.
        top_N (int): Number of top priority levels to keep.
    Returns:
        float: Normalized final value.
    """
    max_size = row_count * column_count

    # Initial valid moves mask
    valid_moves = get_valid_moves_nb(state, row_count, column_count)
    # Pre-filter by priority
    valid_moves = filter_top_priority_moves(
        valid_moves, priority_grid, row_count, column_count, top_N
    )
    total_valid = np.sum(valid_moves)

    # Rollout until no moves remain
    while total_valid > 0:
        acts = np.empty(total_valid, np.int64)
        k = 0
        for idx in range(max_size):
            if valid_moves[idx]:
                acts[k] = idx
                k += 1

        pick = acts[np.random.randint(0, total_valid)]

        # Update valid moves and state
        valid_moves = get_valid_moves_subset_nb(
            state, valid_moves, pick, row_count, column_count
        )
        state[pick // column_count, pick % column_count] = 1

        # Filter again by priority
        valid_moves = filter_top_priority_moves(
            valid_moves, priority_grid, row_count, column_count, top_N
        )
        total_valid = np.sum(valid_moves)

    return get_value_nb(state, pts_upper_bound)

### Bit-pack utilities

In [53]:
import numpy as np

# ========= Bit-pack utilities =========
def _pack_bits_bool2d(arr2d: np.ndarray) -> np.ndarray:
    """
    Pack a 2D 0/1 or bool array into a 1D uint8 bit vector using bitorder='big'.
    """
    # Ensure uint8 0/1
    a = arr2d.astype(np.uint8, copy=False)
    return np.packbits(a.reshape(-1), bitorder='big')

def _unpack_bits_to_2d(bits: np.ndarray, rows: int, cols: int) -> np.ndarray:
    """
    Unpack a 1D uint8 bit vector to a 2D uint8 array (0/1) with given shape.
    """
    flat = np.unpackbits(bits, bitorder='big')
    need = rows * cols
    if flat.size > need:
        flat = flat[:need]
    return flat.reshape((rows, cols)).astype(np.uint8, copy=False)

def _bit_clear_inplace(bits: np.ndarray, idx: int) -> None:
    """
    Clear (set to 0) the bit at flat index idx in the packed array (bitorder='big').
    Uses a non-negative mask to avoid OverflowError from bitwise NOT on Python ints.
    """
    byte_i = idx // 8
    off    = idx % 8
    # Build a clear mask: 0xFF with target bit cleared
    clear_mask = np.uint8(0xFF ^ (1 << (7 - off)))
    bits[byte_i] &= clear_mask

def _bit_set_inplace(bits: np.ndarray, idx: int) -> None:
    """
    Set (to 1) the bit at flat index idx in the packed array (bitorder='big').
    """
    byte_i = idx // 8
    off    = idx % 8
    bits[byte_i] |= np.uint8(1 << (7 - off))

## Node

In [54]:
import math
import numpy as np
import torch
import threading

class Node_AZ:
    def __init__(self, game, args, state, parent=None, action_taken=None, prior=0.0, visit_count=0):
        self.game = game
        self.args = args
        self.state = state
        self.parent = parent
        self.action_taken = action_taken
        self.prior = float(prior)

        self.children = []
        self.visit_count = visit_count
        self.value_sum = 0
        self.lock = threading.Lock()
        self._vl = args.get('virtual_loss', 1.0)

        if parent is None:
            self.level = int(np.sum(state))  # Level is the number of points placed
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_with_symmetry')):
                self.action_space = game.get_valid_moves(state)
                self.valid_moves = game.filter_valid_moves_by_symmetry(
                    self.action_space, state
                ).copy()
            else:
                self.valid_moves = game.get_valid_moves(state)
                self.action_space = self.valid_moves.copy()
        else:
            self.level = parent.level + 1
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_subset_with_symmetry')):
                self.action_space = game.get_valid_moves_subset(
                    parent.state, parent.action_space, self.action_taken)
                self.valid_moves = game.filter_valid_moves_by_symmetry(
                    self.action_space, state
                ).copy()
            else:
                self.valid_moves = game.get_valid_moves_subset(
                    parent.state, parent.action_space, self.action_taken)
                self.action_space = self.valid_moves.copy()
        
        # Ensure action_space is immutable
        self.action_space.flags.writeable = False

        self.is_full = False
        self._cached_ucb = None     # Cached UCB value
        self._ucb_dirty = True      # Indicates whether the cached UCB is stale
    
    # Keep for compatibility, but not used in single thread AlphaZero
    def apply_virtual_loss(self):
        with self.lock:
            self.value_sum -= self._vl
            self.visit_count += 1
            self._ucb_dirty = True

    def revert_virtual_loss(self):
        with self.lock:
            self.value_sum += self._vl
            self._ucb_dirty = True

    def is_fully_expanded(self):
        return self.is_full

    def q_value(self):
        return 0.0 if self.visit_count == 0 else self.value_sum / self.visit_count

    # PUCT formula
    def get_ucb(self, child, iter):
        parent_visit = max(1, self.visit_count)
        q = child.q_value()
        if self.args['exploration_decay'] and iter is not None:
            c = self.args['C'] * exploration_decay_nb(iter/self.args['num_searches'])
        else:
            c = self.args['C']
        # Consider try different exloration functions
        u = c * child.prior * math.sqrt(math.log(parent_visit)) / (1 + child.visit_count)
        return q + u
    
    def select(self, iter=None):
        best_child = None
        best_score = -1e18
        for child in self.children:
            score = self.get_ucb(child, iter)
            if score > best_score:
                best_score = score
                best_child = child
        return best_child

    # legacy expand, not used in AlphaZero
    def expand(self):
        valid_indices = np.where(self.valid_moves == 1)[0]
        action = np.random.choice(valid_indices)
        self.valid_moves[action] = 0

        if np.sum(self.valid_moves) == 0:
            self.is_full = True

        child_state = self.state.copy()
        child_state = self.game.get_next_state(child_state, action)

        child = Node(self.game, self.args, child_state, self, action)
        self.children.append(child)

        return child

    # New expand function using policy vector
    def expand_with_policy(self, policy_vec: np.ndarray):

        if self.is_full:
            return
        
        for action, prob in enumerate(policy_vec):
            if prob > 0 and self.valid_moves[action] == 1:
                child_state = self.state.copy()
                child_state = self.game.get_next_state(child_state, action)
                child = Node_AZ(self.game, self.args, child_state, self, action, prior=prob)
                self.children.append(child)

        self.is_full = True

        if len(self.children) == 0:
            self.is_full = True
    
    # Keep for compatibility, but not used in AlphaZero
    def simulate(self):
        tmp = self.state.copy()
        if self.args.get("simulate_with_priority", False):
            return simulate_with_priority_nb(
                tmp,
                self.game.row_count,
                self.game.column_count,
                self.game.pts_upper_bound,
                self.game.priority_grid,
                self.args['TopN']
            )
        else:
            return simulate_nb(
                tmp,
                self.game.row_count,
                self.game.column_count,
                self.game.pts_upper_bound
            )


    def backpropagate(self, value):
        with self.lock:
            self.value_sum += value
            self._ucb_dirty = True  # Mark UCB as outdated
            self.visit_count += 1
        if self.parent is not None:
            self.parent.backpropagate(value)

## Node Compressed (Working)

In [55]:
import math
import numpy as np
import threading
import concurrent.futures # For parallel expansion

class Node_Compressed_AZ:
    """
    AlphaZero compatible Node that implements bit-packing for memory efficiency.
    - Stores state / valid_moves / action_space as bit-packed arrays.
    - Provides properties (.state, .valid_moves) to return unpacked views for compatibility.
    - Aligned with Node_AZ for use in an AlphaZero MCTS search.
    """
    __slots__ = (
        'game', 'args', 'parent', 'action_taken', 'prior',
        'children', 'visit_count', 'value_sum', 'lock', '_vl',
        'level', 'is_full',
        # packed payloads
        '_rows', '_cols', '_state_bits', '_valid_bits', '_action_bits'
    )

    def __init__(self, game, args, state, parent=None, action_taken=None, prior=0.0, visit_count=0):
        self.game = game
        self.args = args
        self.parent = parent
        self.action_taken = action_taken
        self.prior = float(prior)

        self.children = []
        self.visit_count = visit_count
        self.value_sum = 0.0
        self.lock = threading.Lock()
        self._vl = args.get('virtual_loss', 1.0)

        self._rows = getattr(game, 'row_count', state.shape[0])
        self._cols = getattr(game, 'column_count', state.shape[1] if state.ndim > 1 else self._rows)

        self._state_bits = _pack_bits_bool2d(state)

        if parent is None:
            self.level = int(np.sum(state))
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_with_symmetry')):
                action_space = game.get_valid_moves(state)
                valid_moves  = game.filter_valid_moves_by_symmetry(action_space, state).copy()
            else:
                valid_moves  = game.get_valid_moves(state)
                action_space = valid_moves.copy()
        else:
            self.level = parent.level + 1
            parent_state = parent.state
            parent_action_space = parent.action_space
            if (self.level <= game.max_level_to_use_symmetry and 
                hasattr(game, 'get_valid_moves_subset_with_symmetry')):
                action_space = game.get_valid_moves_subset(parent_state, parent_action_space, self.action_taken)
                valid_moves  = game.filter_valid_moves_by_symmetry(action_space, self.state).copy()
            else:
                valid_moves  = game.get_valid_moves_subset(parent_state, parent_action_space, self.action_taken)
                action_space = valid_moves.copy()

        self._valid_bits  = _pack_bits_bool2d(valid_moves.reshape(self._rows, self._cols))
        self._action_bits = _pack_bits_bool2d(action_space.reshape(self._rows, self._cols))

        self.is_full = False

    @property
    def state(self) -> np.ndarray:
        return _unpack_bits_to_2d(self._state_bits, self._rows, self._cols)

    @property
    def valid_moves(self) -> np.ndarray:
        vm = _unpack_bits_to_2d(self._valid_bits, self._rows, self._cols).reshape(-1)
        vm.flags.writeable = False
        return vm

    @property
    def action_space(self) -> np.ndarray:
        am = _unpack_bits_to_2d(self._action_bits, self._rows, self._cols).reshape(-1)
        am.flags.writeable = False
        return am

    def apply_virtual_loss(self):
        with self.lock:
            self.value_sum -= self._vl
            self.visit_count += 1

    def revert_virtual_loss(self):
        with self.lock:
            self.value_sum += self._vl

    def is_fully_expanded(self):
        return self.is_full

    def q_value(self):
        return 0.0 if self.visit_count == 0 else self.value_sum / self.visit_count

    def get_ucb(self, child, iter):
        parent_visit = max(1, self.visit_count)
        q = child.q_value()
        
        if self.args.get('exploration_decay', False):
            c = self.args['C'] * exploration_decay_nb(iter / self.args['num_searches'])
        else:
            c = self.args['C']
            
        u = c * child.prior * math.sqrt(math.log(parent_visit)) / (1 + child.visit_count)
        return q + u

    def select(self, iter):
        best_child = None
        best_score = -1e18
        for child in self.children:
            score = self.get_ucb(child, iter)
            if score > best_score:
                best_score = score
                best_child = child
        return best_child

    def expand_with_policy(self, policy_vec: np.ndarray):
        if self.is_full:
            return

        def _create_child(action_prob_tuple):
            action, prob = action_prob_tuple
            child_state = self.state.copy()
            child_state = self.game.get_next_state(child_state, action)
            return Node_Compressed_AZ(self.game, self.args, child_state, self, action, prior=prob)

        tasks = []
        current_valid_moves = self.valid_moves
        for action, prob in enumerate(policy_vec):
            if prob > 0 and current_valid_moves[action] == 1:
                tasks.append((action, prob))

        num_workers = self.args.get('num_workers', 1)
        if num_workers > 1 and len(tasks) > 1:
            with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
                self.children = list(executor.map(_create_child, tasks))
        else:
            self.children = [_create_child(task) for task in tasks]

        self.is_full = True

    def backpropagate(self, value):
        node = self
        while node is not None:
            with node.lock:
                node.value_sum += value
                node.visit_count += 1
            node = node.parent

## MCTS

### MCTSVisualizer

In [56]:
import os
import io
import json
import base64
import datetime
import numpy as np
import matplotlib.pyplot as plt
from pyvis.network import Network

# Assume Node class is defined elsewhere

class MCTSVisualizer:
    # Global storage for all trials and steps (class variable)
    global_trial_data = []

    def __init__(self, game, args):
        self.game = game
        self.args = args
        self.snapshots = []  # Store snapshots for the current trial
        self.trial_id = 'unknown'

    def start_new_trial(self, trial_id):
        """Prepares the visualizer for a new trial."""
        self.trial_id = trial_id
        self.snapshots = [] # Clear snapshots from the previous trial

    def _state_to_image_base64(self, state):
        """
        Convert a game state to a base64-encoded image using the game's display_state method.
        """
        plt.figure(figsize=(4, 4))
        rows, cols = self.game.row_count, self.game.column_count
        y_idx, x_idx = np.nonzero(state)
        y_disp = rows - 1 - y_idx
        plt.scatter(x_idx, y_disp, s=200, c='blue', linewidths=0.5)
        plt.xticks(range(cols))
        plt.yticks(range(rows))
        plt.grid(True, alpha=0.3)
        plt.xlim(-0.5, cols - 0.5)
        plt.ylim(-0.5, rows - 0.5)
        plt.gca().set_aspect('equal')
        plt.xticks([])
        plt.yticks([])
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight', dpi=80, pad_inches=0.1)
        buf.seek(0)
        img_base64 = base64.b64encode(buf.read()).decode()
        plt.close()
        return f"data:image/png;base64,{img_base64}"

    def _get_node_label(self, node, iter_num=None):
        """
        Generate a label for a node showing its statistics.
        """
        avg_value = node.value_sum / node.visit_count if node.visit_count > 0 else 0
        ucb = 0
        if node.parent is not None and node.visit_count > 0:
            try:
                ucb = node.parent.get_ucb(node, iter_num or 0)
            except:
                ucb = 0 # Failsafe
        
        label = f"Visits: {node.visit_count}\n"
        label += f"Value Sum: {node.value_sum:.3f}\n"
        label += f"Avg Value: {avg_value:.3f}\n"
        label += f"UCB: {ucb:.3f}"
        return label

    def create_tree_snapshot(self, root, snapshot_name="MCTS Tree"):
        """
        Create a tree visualization snapshot for the current step.
        """
        # (This is your tree_visualization method, renamed for clarity)
        net = Network(height="600px", width="100%", bgcolor="#222222", font_color="white", directed=True)
        net.barnes_hut()
        
        all_nodes, visited = [], set()
        def collect_nodes_dfs(node, level=0):
            if id(node) in visited: return
            visited.add(id(node))
            all_nodes.append((node, level))
            for child in node.children:
                collect_nodes_dfs(child, level + 1)
        
        collect_nodes_dfs(root)
        print(f"Tree visualization: Found {len(all_nodes)} nodes total")

        json_nodes, json_edges, node_mapping = [], [], {}
        for i, (node, level) in enumerate(all_nodes):
            current_id = f"node_{i}"
            node_mapping[id(node)] = current_id
            img_base64 = self._state_to_image_base64(node.state)
            label = self._get_node_label(node)
            
            color = "#4CAF50"  # Default green
            if node.is_fully_expanded(): color = "#2196F3"
            elif len(node.children) == 0 and not node.is_fully_expanded(): color = "#FF9800"
            elif np.sum(node.valid_moves) == 0: color = "#F44336"

            label_lines = label.split('\n')
            escaped_label = label.replace('\n', '\\n')
            title_text = f"Action: {node.action_taken}\\n{escaped_label}\\nChildren: {len(node.children)}\\nValid moves left: {np.sum(node.valid_moves)}"
            
            json_nodes.append({
                "id": current_id, "label": label_lines, "image": img_base64,
                "shape": "image", "size": 30, "level": level, "color": color,
                "title": title_text, "x": i * 100, "y": level * 150
            })
        
        for node, _ in all_nodes:
            current_id = node_mapping[id(node)]
            for child in node.children:
                if id(child) in node_mapping:
                    child_id = node_mapping[id(child)]
                    json_edges.append({
                        "from": current_id, "to": child_id,
                        "smooth": {"type": "cubicBezier", "forceDirection": "vertical", "roundness": 0.4}
                    })
        
        snapshot_data = {
            'name': snapshot_name, 'trial_id': self.trial_id,
            'step_number': len(self.snapshots), 'total_nodes': len(all_nodes),
            'args': self.args.copy(), 'json_nodes': json_nodes, 'json_edges': json_edges
        }
        
        self.snapshots.append(snapshot_data)
        MCTSVisualizer.global_trial_data.append(snapshot_data)
        print(f"Snapshot created for Trial {self.trial_id}, Step {len(self.snapshots)}")

    @classmethod
    def clear_global_data(cls):
        """Clear all global trial data."""
        cls.global_trial_data.clear()
        print("Global trial data cleared.")

    @classmethod
    def save_final_visualization(cls, web_viz_dir=None, experiment_name="mcts_experiment"):
        """
        Save the final comprehensive visualization at the end of all trials.
        """
        # (This is your original save_final_visualization method)
        if not cls.global_trial_data:
            print("No global trial data to save.")
            return None
        if web_viz_dir is None:
            web_viz_dir = './web_visualization'
        os.makedirs(web_viz_dir, exist_ok=True)
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = os.path.join(web_viz_dir, f"{experiment_name}_comprehensive_{timestamp}.html")
        cls._save_comprehensive_html(filename)
        return filename

    @classmethod
    def _save_comprehensive_html(cls, filename="mcts_comprehensive_visualization.html"):
        """
        Creates the comprehensive HTML file with all trials and steps.
        """
        # (This is your original save_comprehensive_html method)
        if not cls.global_trial_data:
            print("No global trial data to save.")
            return

        json_snapshots = []
        for snapshot in cls.global_trial_data:
            json_snapshots.append({
                "id": f"t{snapshot['trial_id']}_s{snapshot['step_number']}",
                "title": f"Trial {snapshot['trial_id']} - {snapshot['name']}",
                "trial_id": snapshot['trial_id'],
                "step_number": snapshot['step_number'],
                "total_nodes": snapshot['total_nodes'],
                "nodes": snapshot.get('json_nodes', []),
                "edges": snapshot.get('json_edges', []),
                "grid_size": snapshot.get('args', {}).get('n', 'unknown')
            })
        
        # Save the JSON data separately for debugging and external use
        json_filename = filename.replace('.html', '_data.json')
        with open(json_filename, 'w', encoding='utf-8') as f:
            json.dump(json_snapshots, f, indent=2, ensure_ascii=False)
        print(f"JSON data saved to: {json_filename}")

        # The large HTML string template goes here. It's omitted for brevity but is identical
        # to the one in your original code.
        html_content = f"""<!doctype html>
<html>
<head>
  <meta charset="utf-8" />
  <title>MCTS Comprehensive Tree Visualization</title>
  <style>
    /* ... Your CSS styles ... */
  </style>
</head>
<body>
  <script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
  <script>
    const snapshots = {json.dumps(json_snapshots, ensure_ascii=False, indent=2)};
    // ... The rest of your JavaScript for navigation, rendering, etc. ...
  </script>
</body>
</html>
"""
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(html_content)
        print(f"Comprehensive visualization saved to: {filename}")

### MCTS (Not for AlphaZero)

In [57]:

from tqdm import trange

class MCTS:
    def __init__(self, game, args={
        'num_searches': 1000,
        'C': 1.4,
        'tree_visualization': False # Control visualization from args
    }):
        self.game = game
        self.args = args
        self.trial_id = None # Current trial ID

        # If visualization is enabled, create a visualizer instance
        if self.args.get('tree_visualization', False):
            self.visualizer = MCTSVisualizer(self.game, self.args)
        else:
            self.visualizer = None

    def start_new_trial(self, trial_id):
        """Starts a new trial, like a new game or experiment run."""
        self.trial_id = trial_id
        if self.visualizer:
            self.visualizer.start_new_trial(trial_id)

    def search(self, state):
        # define root
        if self.args.get('node_compression', False):
            root = Node_Compressed(self.game, self.args, state)
            print("Using Node_Compressed for MCTS")
        else:
            root = Node(self.game, self.args, state)

        if self.args['process_bar'] == True:
            search_iterator = trange(self.args['num_searches'])
        else:
            search_iterator = range(self.args['num_searches'])

        for search in search_iterator:
            node = root

            # selection
            while node.is_fully_expanded(): #         return self.is_full and len(self.children) > 0
                node = node.select(iter=search)

            if node.action_taken is not None:
                value, is_terminal = self.game.get_value_and_terminated(node.state, node.valid_moves)
                # has_collinear = self.game.check_collinear(node.state, node.action_taken)
                # value, _ = self.game.get_value_and_terminated(node.state)

                if not is_terminal:
                    node = node.expand()
                    value = node.simulate()
            else:
                node = node.expand()
                value = node.simulate()

            node.backpropagate(value)

        action_probs = np.zeros(self.game.action_size)
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        action_probs /= np.sum(action_probs)
        
        # ---- DELEGATE VISUALIZATION ----
        if self.visualizer:
            num_points = np.sum(state)
            snapshot_name = f"Step {num_points}: {num_points} points placed"
            
            # Call the visualizer to create the snapshot
            self.visualizer.create_tree_snapshot(root, snapshot_name)
            
            if self.args.get('pause_at_each_step', False):
                try:
                    response = input("Output action prob? (y/n): ").strip().lower()
                    if response == 'y':
                        print("Action probabilities:", action_probs)
                except (EOFError, KeyboardInterrupt):
                    pass
        
        return action_probs

## AlphaZero MCTS

In [58]:
from tqdm import trange
import torch

class MCTS_AZ:
    def __init__(self, game, args, model):
        self.game = game
        self.model = model
        self.args = args
        self.trial_id = None # Current trial ID

        # If visualization is enabled, create a visualizer instance
        if self.args.get('tree_visualization', False):
            self.visualizer = MCTSVisualizer(self.game, self.args)
        else:
            self.visualizer = None

    def start_new_trial(self, trial_id):
        """Starts a new trial, like a new game or experiment run."""
        self.trial_id = trial_id
        if self.visualizer:
            self.visualizer.start_new_trial(trial_id)

    @torch.no_grad()
    def _infer(self, state):
        encoded_state = self.game.get_encoded_state(state)  # Game must provide encoding
        state_tensor = torch.tensor(encoded_state, dtype=torch.float32, device=self.model.device).unsqueeze(0)
        logits, value = self.model(state_tensor)
        policy = torch.softmax(logits, dim=1).squeeze(0).cpu().numpy()
        return policy, float(value.item())

    @torch.no_grad()
    def search(self, state):
        # define root
        if self.args.get('node_compression', False):
            root = Node_Compressed_AZ(self.game, self.args, state)
            print("Using Node_Compressed_AZ for MCTS")
        else:
            root = Node_AZ(self.game, self.args, state)

        # Root inference
        policy, value = self._infer(state)
        
        # Masking
        policy *= root.valid_moves
        policy_sum = policy.sum()
        if policy_sum <= 0:
            # Uniform over legal moves if policy is all zero
            valid_moves = root.valid_moves
            valid_moves_count = valid_moves.sum()
            if valid_moves_count > 0:
                policy = valid_moves / valid_moves_count
        else:
            policy /= policy_sum

        # Add Dirichlet noise for exploration at the root
        eps = self.args.get('dirichlet_epsilon', 0.0)
        if eps > 0:
            alpha = self.args.get('dirichlet_alpha', 0.3)
            noise = np.random.dirichlet([alpha] * self.game.action_size)
            policy = (1 - eps) * policy + eps * noise
            # Re-mask and normalize
            policy *= root.valid_moves
            policy_sum_after_noise = policy.sum()
            if policy_sum_after_noise > 0:
                policy /= policy_sum_after_noise

        root.expand_with_policy(policy)
        root.backpropagate(value)

        if self.args.get('process_bar', True):
            search_iterator = trange(self.args['num_searches'])
        else:
            search_iterator = range(self.args['num_searches'])

        for i in search_iterator:
            node = root
            # Selection
            while node.is_fully_expanded() and len(node.children) > 0:
                node = node.select(iter=i)

            # Terminal state check
            term_value, terminal = self.game.get_value_and_terminated(node.state, node.valid_moves)
            if terminal:
                node.backpropagate(term_value)
                continue

            # Infer and expand
            policy, value = self._infer(node.state)
            policy *= node.valid_moves
            policy_sum = policy.sum()
            if policy_sum <= 0:
                valid_moves = node.valid_moves
                valid_moves_count = valid_moves.sum()
                if valid_moves_count > 0:
                    policy = valid_moves / valid_moves_count
            else:
                policy /= policy_sum
            
            node.expand_with_policy(policy)
            node.backpropagate(value)

        # Calculate action probabilities based on visit counts
        action_probs = np.zeros(self.game.action_size, dtype=np.float32)
        for child in root.children:
            action_probs[child.action_taken] = child.visit_count
        
        total_visits = action_probs.sum()
        if total_visits > 0:
            action_probs /= total_visits
        
        # ---- DELEGATE VISUALIZATION ----
        if self.visualizer:
            num_points = np.sum(state)
            snapshot_name = f"Step {num_points}: {num_points} points placed"
            
            # Call the visualizer to create the snapshot
            self.visualizer.create_tree_snapshot(root, snapshot_name)
            
            if self.args.get('pause_at_each_step', False):
                try:
                    response = input("Output action prob? (y/n): ").strip().lower()
                    if response == 'y':
                        print("Action probabilities:", action_probs)
                except (EOFError, KeyboardInterrupt):
                    pass
        
        return action_probs

## AlphaZero

In [ ]:
# AlphaZero training loop for single-player N3il optimization
import numpy as np
import random
import torch
import torch.nn.functional as F
from tqdm import trange

class AlphaZero:
    """
    AlphaZero with optional post-batch GPU data augmentation (D4).
    Set args:
        data_augmentation_per_traj=False            # disable per-episode CPU aug
        post_batch_augmentation=True       # enable one-shot GPU aug after self-play
        keep_aug_on_device=True            # keep augmented tensors on device (no CPU round trip)
    """
    def __init__(self, model, optimizer, game, args):
        self.model = model
        self.optimizer = optimizer
        self.game = game
        self.args = args
        self.mcts = MCTS_AZ(game, args, model)

        seed = args.get('random_seed', None)
        if seed is not None:
            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)

        self.policy_loss_coef = args.get('policy_loss_coef', 1.0)
        self.value_loss_coef  = args.get('value_loss_coef', 1.0)
        self.gradient_clip    = args.get('gradient_clip', None)

        # CPU numpy symmetry funcs (legacy / fallback per-trajectory)
        self._symmetry_funcs = [
            ("id",        lambda x: x),
            ("rot90",     lambda x: np.rot90(x, 1)),
            ("rot180",    lambda x: np.rot90(x, 2)),
            ("rot270",    lambda x: np.rot90(x, 3)),
            ("flip_h",    lambda x: np.flipud(x)),
            ("flip_v",    lambda x: np.fliplr(x)),
            ("flip_diag", lambda x: x.T),
            ("flip_anti", lambda x: np.rot90(x.T, 2)),
        ]
        # Torch batch transforms (operate on (B,C,n,n))
        self._torch_tf = [
            ("id",        lambda t: t),
            ("rot90",     lambda t: torch.rot90(t, 1, (-2, -1))),
            ("rot180",    lambda t: torch.rot90(t, 2, (-2, -1))),
            ("rot270",    lambda t: torch.rot90(t, 3, (-2, -1))),
            ("flip_h",    lambda t: torch.flip(t, (-2,))),         # vertical axis in image => flip rows
            ("flip_v",    lambda t: torch.flip(t, (-1,))),         # horizontal axis => flip cols
            ("flip_diag", lambda t: t.transpose(-2, -1)),
            ("flip_anti", lambda t: torch.rot90(t.transpose(-2, -1), 2, (-2, -1))),
        ]
        self._policy_perms = None  # (8, A) long tensor of flattened index permutations

    def _apply_temperature(self, action_probs, step_index):
        temp = self.args.get('temperature', 1.0)
        schedule = self.args.get('temperature_schedule')
        if callable(schedule):
            temp = schedule(step_index)
        if temp <= 0:
            greedy = np.zeros_like(action_probs)
            greedy[np.argmax(action_probs)] = 1.0
            return greedy
        p = np.power(action_probs, 1.0 / temp)
        s = p.sum()
        if s <= 0:
            return np.ones_like(p) / len(p)
        return p / s

    # Per-trajectory CPU augmentation (kept for backward compatibility)
    def _augment_trajectory(self, trajectory):
        augmented = []
        for encoded, policy_vec, value in trajectory:
            C, n, _ = encoded.shape
            policy_2d = policy_vec.reshape(n, n)
            for name, tf in self._symmetry_funcs:
                enc_tf = np.stack([tf(encoded[c]) for c in range(C)], axis=0)
                pol_tf = tf(policy_2d).reshape(-1)
                augmented.append((enc_tf, pol_tf, value))
        return augmented

    def _build_policy_perms(self, n, device):
        base = np.arange(n * n).reshape(n, n)
        perms = []
        for name, tf in self._symmetry_funcs:
            perms.append(tf(base).reshape(-1))
        perm_np = np.stack(perms, axis=0)  # (8, A)
        self._policy_perms = torch.as_tensor(perm_np, dtype=torch.long, device=device)

    def _augment_memory_batch(self, memory):
        """
        GPU batch augmentation.
        memory: list of (encoded_state(C,n,n), policy_vec(A), value)
        Returns list with 8x entries. States/policies may remain as torch.Tensor if keep_aug_on_device.
        """
        if not memory:
            return memory
        device = self.model.device
        states = torch.tensor(
            np.stack([s for s, _, _ in memory], axis=0),
            dtype=torch.float32, device=device
        )  # (B,C,n,n)
        policies = torch.tensor(
            np.stack([p for _, p, _ in memory], axis=0),
            dtype=torch.float32, device=device
        )  # (B,A)
        values = torch.tensor(
            [v for _, _, v in memory],
            dtype=torch.float32, device=device
        )  # (B,)

        B, C, n, _ = states.shape
        A = n * n
        if self._policy_perms is None or self._policy_perms.shape[1] != A:
            self._build_policy_perms(n, device)

        aug_states = []
        aug_pols = []
        for idx, (name, tf) in enumerate(self._torch_tf):
            st = tf(states)
            perm = self._policy_perms[idx]
            pol = policies.index_select(1, perm)
            aug_states.append(st)
            aug_pols.append(pol)

        aug_states = torch.stack(aug_states, 1).reshape(B * 8, C, n, n)
        aug_pols   = torch.stack(aug_pols, 1).reshape(B * 8, A)
        aug_vals   = values.repeat_interleave(8)

        if self.args.get('keep_aug_on_device', True):
            return [(aug_states[i], aug_pols[i], aug_vals[i].item()) for i in range(aug_states.size(0))]

        # Move back to CPU numpy if requested
        aug_states_np = aug_states.cpu().numpy()
        aug_pols_np   = aug_pols.cpu().numpy()
        aug_vals_np   = aug_vals.cpu().numpy()
        return [(aug_states_np[i], aug_pols_np[i], aug_vals_np[i]) for i in range(aug_states_np.shape[0])]

    def self_play(self):
        """
        Run one full episode.
        Returns list of (encoded_state, policy_target, value_target).
        (No augmentation here if using post_batch_augmentation.)
        """
        memory = []
        state = self.game.get_initial_state()
        step = 0
        while True:
            action_probs = self.mcts.search(state)
            stored_state = state.copy()
            memory.append((stored_state, action_probs))
            tau_policy = self._apply_temperature(action_probs, step)
            action = np.random.choice(self.game.action_size, p=tau_policy)
            state = self.game.get_next_state(state.copy(), action)
            valid_moves = self.game.get_valid_moves(state)
            value, terminal = self.game.get_value_and_terminated(state, valid_moves)
            if terminal:
                trajectory = []
                for hist_state, hist_policy in memory:
                    encoded = self.game.get_encoded_state(hist_state)
                    trajectory.append((encoded, hist_policy, value))
                # Per-trajectory CPU augmentation only if enabled and not using batch aug
                if self.args.get('data_augmentation_per_traj', False) and not self.args.get('post_batch_augmentation', False):
                    trajectory = self._augment_trajectory(trajectory)
                return trajectory
            step += 1

    def _policy_value_loss(self, logits, values, policy_targets, value_targets):
        log_probs = F.log_softmax(logits, dim=1)
        policy_loss = -(policy_targets * log_probs).sum(dim=1).mean()
        value_loss = F.mse_loss(values, value_targets)
        loss = self.policy_loss_coef * policy_loss + self.value_loss_coef * value_loss
        return loss, policy_loss.item(), value_loss.item()

    def train_epoch(self, memory):
        random.shuffle(memory)
        batch_size = self.args.get('batch_size', 32)
        self.model.train()
        metrics = []
        for start in range(0, len(memory), batch_size):
            batch = memory[start:start + batch_size]
            if not batch:
                continue
            states, policy_targets, value_targets = zip(*batch)

            # Support both numpy arrays and pre-loaded torch tensors
            if isinstance(states[0], torch.Tensor):
                states_t = torch.stack(states, 0).to(self.model.device)
            else:
                states_t = torch.tensor(np.array(states), dtype=torch.float32, device=self.model.device)

            if isinstance(policy_targets[0], torch.Tensor):
                policy_t = torch.stack(policy_targets, 0).to(self.model.device)
            else:
                policy_t = torch.tensor(np.array(policy_targets), dtype=torch.float32, device=self.model.device)

            value_t = torch.tensor(np.array(value_targets).reshape(-1, 1),
                                   dtype=torch.float32, device=self.model.device)

            logits, pred_values = self.model(states_t)
            loss, pl, vl = self._policy_value_loss(logits, pred_values, policy_t, value_t)
            self.optimizer.zero_grad()
            loss.backward()
            if self.gradient_clip is not None:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.gradient_clip)
            self.optimizer.step()
            metrics.append((loss.item(), pl, vl))

        if metrics:
            return {
                'loss': np.mean([m[0] for m in metrics]),
                'policy_loss': np.mean([m[1] for m in metrics]),
                'value_loss': np.mean([m[2] for m in metrics]),
            }
        return {}

    def learn(self):
        num_iterations = self.args.get('num_iterations', 1)
        num_selfplay   = self.args.get('num_selfPlay_iterations', 1)
        num_epochs     = self.args.get('num_epochs', 1)
        save_interval  = self.args.get('save_interval', 1)
        weight_prefix  = self.args.get('weight_file_name', 'alphazero')

        import os
        weights_dir = self.args.get('weights_dir', None)
        if weights_dir is not None:
            os.makedirs(weights_dir, exist_ok=True)

        all_stats = []
        for it in range(num_iterations):
            self.model.eval()
            memory = []
            for _ in trange(num_selfplay, desc=f"SelfPlay Iter {it+1}/{num_iterations}", leave=False):
                memory.extend(self.self_play())

            if self.args.get('post_batch_augmentation', False):
                memory = self._augment_memory_batch(memory)

            epoch_stats = []
            for ep in trange(num_epochs, desc=f"Train Iter {it+1}/{num_iterations}", leave=False):
                stats = self.train_epoch(memory)
                epoch_stats.append(stats)
            all_stats.append(epoch_stats)

            if (it + 1) % save_interval == 0:
                if weights_dir is None:
                    model_path = f"{weight_prefix}_model_{it+1}.pt"
                    opt_path   = f"{weight_prefix}_optimizer_{it+1}.pt"
                else:
                    model_path = os.path.join(weights_dir, f"{weight_prefix}_model_{it+1}.pt")
                    opt_path   = os.path.join(weights_dir, f"{weight_prefix}_optimizer_{it+1}.pt")
                torch.save(self.model.state_dict(), model_path)
                torch.save(self.optimizer.state_dict(), opt_path)
                if self.args.get('logging_mode', False):
                    print(f"Saved: {model_path}  {opt_path}")
        return all_stats

    @torch.no_grad()
    def inference_step(self, state, temperature: float = 0.0, return_net: bool = True):
        """
        Single-step inference helper.
        Args:
            state (np.ndarray): raw game state.
            temperature (float): sampling temperature (0 => argmax).
            return_net (bool): if True also return raw network policy/value.
        Returns:
            (action, mcts_policy[, net_policy, net_value])
        """
        old_pb = self.args.get('process_bar', False)
        old_eps = self.args.get('dirichlet_epsilon', 0.0)
        self.args['process_bar'] = False
        self.args['dirichlet_epsilon'] = 0.0

        mcts_policy = self.mcts.search(state)

        self.args['process_bar'] = old_pb
        self.args['dirichlet_epsilon'] = old_eps

        if temperature <= 0:
            pi = np.zeros_like(mcts_policy)
            pi[np.argmax(mcts_policy)] = 1.0
        else:
            p = np.power(mcts_policy, 1.0 / temperature)
            s = p.sum()
            pi = p / s if s > 0 else np.ones_like(p) / len(p)

        action = int(np.argmax(pi))

        if not return_net:
            return action, mcts_policy

        encoded = self.game.get_encoded_state(state)
        t = torch.tensor(encoded, dtype=torch.float32, device=self.model.device).unsqueeze(0)
        logits, value = self.model(t)
        net_policy = torch.softmax(logits, dim=1).cpu().numpy()[0]
        net_value = float(value.item())
        return action, mcts_policy, net_policy, net_value

## Test

### Smoke Test

In [41]:
def run_alphazero_smoke_test(
    game_class,
    args: dict,
    model_class,
    weight_path: str | None = None,
    n: int | None = None,
    n_res_blocks: int = 3,
    n_hidden: int = 64,
    num_searches: int = 30,
    seed: int | None = 1,
    device: str | None = None,
    show_board: bool = True,
) -> dict:
    """AlphaZero smoke test.

    Attempts both (game, model, args) and (game, args, model) ctor orders.
    Returns dict: raw_policy, raw_value, mcts_policy, search_time_sec.
    """
    import time, copy, os, numpy as np, torch, random, inspect

    test_args = copy.deepcopy(args)
    if n is None:
        n = test_args.get('n')
    test_args.update({'num_searches': num_searches, 'process_bar': False, 'tree_visualization': False})

    if seed is not None:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    game = game_class((n, n), test_args)

    if device is None:
        if torch.backends.mps.is_available(): device = 'mps'
        elif torch.cuda.is_available(): device = 'cuda'
        else: device = 'cpu'
    device = torch.device(device)

    model = model_class(game, num_resBlocks=n_res_blocks, num_hidden=n_hidden, device=device)
    if weight_path is not None:
        if not os.path.isfile(weight_path): raise FileNotFoundError(weight_path)
        model.load_state_dict(torch.load(weight_path, map_location=device))
    model.eval()

    ctor_tried = []
    mcts = None
    # Preferred guesses in order
    orders = [(game, model, test_args), (game, test_args, model)]
    for params in orders:
        try:
            candidate = MCTS_AZ(*params)
            if isinstance(getattr(candidate, 'args', None), dict):
                mcts = candidate
                break
        except Exception as e:
            ctor_tried.append(str(e))
    if mcts is None:
        raise TypeError('Failed to construct MCTS_AZ with tested signatures. Errors: ' + ' | '.join(ctor_tried))

    init_state = game.get_initial_state()
    if show_board and hasattr(game, 'display_state'):
        print('Initial state:'); game.display_state(init_state)

    with torch.no_grad():
        enc = game.get_encoded_state(init_state)
        t = torch.tensor(enc, dtype=torch.float32, device=model.device).unsqueeze(0)
        logits, value = model(t)
        raw_policy = torch.softmax(logits, dim=1).cpu().numpy()[0]
        raw_value = float(value.item())

    start = time.time()
    visit_policy = mcts.search(init_state)
    elapsed = time.time() - start

    if show_board:
        print('\nRaw net policy (reshaped):')
        print(raw_policy.reshape(n, n))
        print(f'Raw value: {raw_value:.4f}')
        print('\nMCTS visit policy (reshaped):')
        print(visit_policy.reshape(n, n))
        print(f'Search time: {elapsed:.3f}s')

    return {
        'raw_policy': raw_policy,
        'raw_value': raw_value,
        'mcts_policy': visit_policy,
        'search_time_sec': elapsed,
    }

# Example:
# res = run_alphazero_smoke_test_v2(N3il, args, ResNet, n=args['n'])
# print(res)

In [60]:
n=5

current_dir = os.getcwd()

# ...existing code...
args = {
    'environment': 'N3il',
    'algorithm': 'AlphaZero',
    'max_level_to_use_symmetry': -1,
    'n': n,
    'C': 1.41,
    'exploration_decay': False,          # keep False unless using decay
    'dirichlet_epsilon': 0,           # >0 to add root noise
    'dirichlet_alpha': 0.3,
    'num_searches': 10*(n**2),                 # per move MCTS simulations
    'num_workers': 1,
    'virtual_loss': 1.0,
    'process_bar': True,
    'display_state': False,
    'logging_mode': True,
    'TopN': 2*n,
    'simulate_with_priority': False,
    'random_seed': 1,
    'tree_visualization': False,
    'node_compression': False,           # set True to use Node_Compressed_AZ
    # ---- AlphaZero training hyperparams ----
    'num_iterations': 10,                # outer loop
    'num_selfPlay_iterations': 100,       # episodes per iteration
    'num_epochs': 5,                     # train epochs per iteration
    'batch_size': 64,
    'temperature': 1.0,                  # sampling temperature
    # 'temperature_schedule': lambda step: 1.0 if step < 4 else 0.1,  # optional
    'policy_loss_coef': 1.0,
    'value_loss_coef': 1.0,
    'gradient_clip': 5.0,
    'save_interval': 1,
    'weight_file_name': 'az_n3',
    # --- Data augmentation ---
    'data_augmentation': False,          # 
    'post_batch_augmentation': True,     # Data Aug in GPU after all self-play
    'keep_aug_on_device': True,          # keep on GPU
    # Optional evaluation / early stop hooks
    # 'early_stop_value_threshold': 0.98,
    # 'eval_interval': 5,
    # Paths
    'weights_dir': os.path.join(current_dir, 'weights'),
    'table_dir': current_dir,
    'figure_dir': os.path.join(current_dir, 'figure'),
    'checkpoint_dir': os.path.join(current_dir, 'checkpoints'),
}



In [61]:
res = run_alphazero_smoke_test(N3il, args, ResNet, n=args['n'])
print(res)

Initial state:
Plot saved as: /Users/luoninz1/063_Research/no-three-in-line/Code/RLMath/tests/test_Alpha_Zero/figure/20250830_165435_5by5/no_three_in_line_5x5_pts0_AlphaZero_20250830_165435.png (DPI: 150)

Raw net policy (reshaped):
[[0.03920263 0.04185543 0.03815395 0.04480958 0.03777303]
 [0.03966169 0.04122015 0.04078608 0.03461931 0.03693119]
 [0.04054327 0.0396292  0.04432408 0.0373699  0.04001862]
 [0.03972886 0.04281408 0.03851572 0.03991892 0.0392952 ]
 [0.04426826 0.03765393 0.04235314 0.04215448 0.03639935]]
Raw value: 0.2134

MCTS visit policy (reshaped):
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
Search time: 0.066s
{'raw_policy': array([0.03920263, 0.04185543, 0.03815395, 0.04480958, 0.03777303,
       0.03966169, 0.04122015, 0.04078608, 0.03461931, 0.03693119,
       0.04054327, 0.0396292 , 0.04432408, 0.0373699 , 0.04001862,
       0.03972886, 0.04281408, 0.03851572, 0.03991892, 0.0392952 ,
       0.04426826, 0.03765393, 0.

### Run AlphaZero

In [ ]:
n=5

current_dir = os.getcwd()

# ...existing code...
args = {
    'environment': 'N3il',
    'algorithm': 'AlphaZero',
    'max_level_to_use_symmetry': -1,
    'n': n,
    'C': 1.41,
    'exploration_decay': False,          # keep False unless using decay
    'dirichlet_epsilon': 0,           # >0 to add root noise
    'dirichlet_alpha': 0.3,
    'num_searches': 10*(n**2),                 # per move MCTS simulations
    'num_workers': 1,
    'virtual_loss': 1.0,
    'process_bar': True,
    'display_state': False,
    'logging_mode': True,
    'TopN': 2*n,
    'simulate_with_priority': False,
    'random_seed': 1,
    'tree_visualization': False,
    'node_compression': False,           # set True to use Node_Compressed_AZ
    # ---- AlphaZero training hyperparams ----
    'num_iterations': 10,                # outer loop
    'num_selfPlay_iterations': 100,       # episodes per iteration
    'num_epochs': 5,                     # train epochs per iteration
    'batch_size': 64,
    'temperature': 1.0,                  # sampling temperature
    # 'temperature_schedule': lambda step: 1.0 if step < 4 else 0.1,  # optional
    'policy_loss_coef': 1.0,
    'value_loss_coef': 1.0,
    'gradient_clip': 5.0,
    'save_interval': 1,
    'weight_file_name': 'az_n3',
    # --- Data augmentation ---
    'data_augmentation': False,          # 
    'post_batch_augmentation': True,     # Data Aug in GPU after all self-play
    'keep_aug_on_device': True,          # keep on GPU
    # Optional evaluation / early stop hooks
    # 'early_stop_value_threshold': 0.98,
    # 'eval_interval': 5,
    # Paths
    'weights_dir': os.path.join(current_dir, 'weights'),
    'table_dir': current_dir,
    'figure_dir': os.path.join(current_dir, 'figure'),
    'checkpoint_dir': os.path.join(current_dir, 'checkpoints'),
}



In [ ]:
# === Run AlphaZero Training ===
import torch, os, numpy as np

# Rebuild environment (uses existing args)
n = args['n']
env = N3il((n, n), args)

# Model, optimizer, (optionally scheduler)
model = ResNet(env, num_resBlocks=3, num_hidden=64, device=device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

az = AlphaZero(model, optimizer, env, args)

print("Start training...")
train_stats = az.learn()
print("Training finished.")

Start training...


100%|██████████| 250/250 [00:00<00:00, 135143.19it/s]


Saved: /Users/luoninz1/063_Research/no-three-in-line/Code/RLMath/tests/test_Alpha_Zero/weights/az_n3_model_1.pt  /Users/luoninz1/063_Research/no-three-in-line/Code/RLMath/tests/test_Alpha_Zero/weights/az_n3_optimizer_1.pt


Train Iter 2/10:  80%|████████  | 4/5 [00:06<00:01,  1.65s/it]       

In [ ]:
# Optional: inspect last iteration average losses
if train_stats:
    last_iter = train_stats[-1]
    if last_iter:
        mean_loss = np.mean([e['loss'] for e in last_iter if e])
        mean_pl   = np.mean([e['policy_loss'] for e in last_iter if e])
        mean_vl   = np.mean([e['value_loss'] for e in last_iter if e])
        print(f"Last iteration mean losses -> total:{mean_loss:.4f} policy:{mean_pl:.4f} value:{mean_vl:.4f}")


### Inference Test

In [ ]:
# === Quick Inference Test (single step) ===
state = env.get_initial_state()
action, mcts_policy, net_policy, net_value = az.inference_step(state, temperature=0.0, return_net=True)
r, c = divmod(action, n)
print(f"Inference action (row={r}, col={c})  net_value={net_value:.4f}")
print("MCTS policy (reshape):")
print(mcts_policy.reshape(n, n))

In [ ]:
# (Optional) Apply the action and show resulting board
next_state = env.get_next_state(state.copy(), action)
if hasattr(env, "display_state"):
    print("Board after action:")
    env.display_state(next_state)

In [ ]:
# === (Optional) Save 'latest' convenience copies ===
weights_dir = args.get('weights_dir')
if weights_dir:
    latest_model = os.path.join(weights_dir, "az_latest_model.pt")
    latest_opt   = os.path.join(weights_dir, "az_latest_optimizer.pt")
    torch.save(model.state_dict(), latest_model)
    torch.save(optimizer.state_dict(), latest_opt)
    print(f"Saved latest checkpoints to {latest_model} / {latest_opt}")